# NSL-KDD Exploratory Data Analysis (Google Colab)
Network Intrusion Detection — EDA notebook, structured to match the written EDA report.

**How to use this in Colab:**
1. Go to https://colab.research.google.com → File → Upload notebook → upload this `.ipynb`.
2. Run the **Setup** cell below and upload `NSL_KDD_70_15_15_Dataset.zip` when prompted (or mount Google Drive instead — both options are given).
3. Run all remaining cells in order (`Runtime → Run all`).


## 1. Setup — get the data into Colab
Two options are given below. Use **whichever one matches where your zip file lives.**

In [ ]:
# Install/confirm libraries (Colab has most of these already)
!pip -q install pandas numpy matplotlib seaborn scikit-learn

### Option A — Upload the zip directly from your computer
Run this cell, then click **Choose Files** and select `NSL_KDD_70_15_15_Dataset.zip`.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select NSL_KDD_70_15_15_Dataset.zip in the dialog

import zipfile, os
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('nsl_kdd_data')

print(os.listdir('nsl_kdd_data'))

### Option B — Load from Google Drive instead
Skip Option A above if you use this. Upload the zip to your Drive first, then:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile
zip_path = '/content/drive/MyDrive/NSL_KDD_70_15_15_Dataset.zip'  # <-- update path
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('nsl_kdd_data')

import os
print(os.listdir('nsl_kdd_data'))

### Load the three splits

In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv('nsl_kdd_data/KDD_train_70.csv')
val   = pd.read_csv('nsl_kdd_data/KDD_val_15.csv')
test  = pd.read_csv('nsl_kdd_data/KDD_test_15.csv')

print('Train:', train.shape)
print('Val:  ', val.shape)
print('Test: ', test.shape)
train.head()

## 2. Dataset Overview

In [ ]:
summary = pd.DataFrame({
    'Split': ['Train (70%)', 'Validation (15%)', 'Test (15%)'],
    'Rows': [train.shape[0], val.shape[0], test.shape[0]],
    'Columns': [train.shape[1], val.shape[1], test.shape[1]],
})
summary

In [ ]:
print('Missing values:', train.isnull().sum().sum())
print('Duplicate rows: ', train.duplicated().sum())

dtypes_summary = train.dtypes.value_counts()
print('\nFeature types:\n', dtypes_summary)

categorical_cols = train.select_dtypes(include='object').columns.tolist()
print('\nCategorical columns:', categorical_cols)

In [ ]:
# Zero-variance / constant features
constant_cols = [c for c in train.columns if train[c].nunique() == 1]
print('Constant (zero-variance) features:', constant_cols)

# Near-binary (0/1) numeric features
near_binary = [c for c in train.select_dtypes(include=np.number).columns
               if set(train[c].unique()).issubset({0, 1})]
print('Near-binary features:', near_binary)

## 3. Target Variable: Attack Categories
The raw `label` column has 23 distinct values (normal + 22 attack names). Group them into the
standard NSL-KDD taxonomy: **Normal, DoS, Probe, R2L, U2R.**

In [ ]:
attack_map = {
    'normal': 'Normal',
    # DoS
    'back': 'DoS', 'land': 'DoS', 'neptune': 'DoS', 'pod': 'DoS', 'smurf': 'DoS', 'teardrop': 'DoS',
    # Probe
    'ipsweep': 'Probe', 'nmap': 'Probe', 'portsweep': 'Probe', 'satan': 'Probe',
    # R2L
    'ftp_write': 'R2L', 'guess_passwd': 'R2L', 'imap': 'R2L', 'multihop': 'R2L',
    'phf': 'R2L', 'spy': 'R2L', 'warezclient': 'R2L', 'warezmaster': 'R2L',
    # U2R
    'buffer_overflow': 'U2R', 'loadmodule': 'U2R', 'perl': 'U2R', 'rootkit': 'U2R',
}

for df in (train, val, test):
    df['attack_category'] = df['label'].map(attack_map)

# sanity check — anything unmapped?
print('Unmapped labels in train:', train.loc[train['attack_category'].isna(), 'label'].unique())

cat_counts = train['attack_category'].value_counts()
cat_pct = (cat_counts / len(train) * 100).round(2)
pd.DataFrame({'Count': cat_counts, '% of Train': cat_pct})

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

order = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=train, x='attack_category', order=order, palette='viridis')
ax.set_yscale('log')  # log scale makes R2L/U2R visible next to Normal/DoS
plt.title('Class Distribution (log scale) — Train Set')
plt.ylabel('Count (log scale)')
plt.xlabel('')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

print(f"Normal:U2R ratio ≈ {cat_counts['Normal'] / cat_counts['U2R']:.0f} : 1")

## 4. Categorical Features

In [ ]:
for col in ['protocol_type', 'flag']:
    print(f'--- {col} ---')
    print((train[col].value_counts(normalize=True) * 100).round(1))
    print()

print('service: number of unique categories =', train['service'].nunique())
print(train['service'].value_counts().head(10))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=train, x='protocol_type', hue='attack_category', hue_order=order, ax=axes[0], palette='viridis')
axes[0].set_title('Protocol Type by Attack Category')
axes[0].set_yscale('log')

top_services = train['service'].value_counts().head(10).index
sns.countplot(data=train[train['service'].isin(top_services)], y='service',
              order=top_services, ax=axes[1], color='#1C7293')
axes[1].set_title('Top 10 Services by Frequency')

plt.tight_layout()
plt.show()

## 5. Numeric Feature Relationships

In [ ]:
numeric_cols = train.select_dtypes(include=np.number).columns.tolist()
corr = train[numeric_cols].corr()

# Find highly correlated pairs (|r| > 0.95), excluding self-correlation
high_corr_pairs = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        r = corr.iloc[i, j]
        if abs(r) > 0.95:
            high_corr_pairs.append((corr.columns[i], corr.columns[j], round(r, 3)))

high_corr_df = pd.DataFrame(high_corr_pairs, columns=['Feature 1', 'Feature 2', 'Correlation']) \
                  .sort_values('Correlation', key=abs, ascending=False)
high_corr_df

In [ ]:
plt.figure(figsize=(16, 12))
sns.heatmap(corr, cmap='coolwarm', center=0, square=True, cbar_kws={'shrink': 0.6})
plt.title('Correlation Heatmap — Numeric Features')
plt.tight_layout()
plt.show()

## 6. Feature Behavior by Class

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col in zip(axes, ['src_bytes', 'dst_bytes', 'count']):
    sns.boxplot(data=train, x='attack_category', y=np.log1p(train[col]), order=order, ax=ax, palette='viridis')
    ax.set_title(f'log1p({col}) by Class')
    ax.set_xlabel('')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=train, x='attack_category', y='serror_rate', order=order, palette='viridis')
plt.title('SYN Error Rate by Class')
plt.tight_layout()
plt.show()

## 7. Implications for Modeling (summary)

| Finding | Implication |
|---|---|
| Severe class imbalance (R2L/U2R) | Stratified sampling, class-weighting, SMOTE; report per-class P/R/F1 — never accuracy alone |
| Highly correlated features (\|r\| > 0.95) | Tree ensembles tolerate this; SVM/KNN/Logistic Regression benefit from de-duplication or PCA |
| `service` has 69 categories | Frequency/target encoding rather than plain one-hot |
| Skewed numeric features (bytes, counts) | `log1p` transform, or use scale-robust tree models as a baseline |
| `num_outbound_cmds` constant | Drop — no signal |

**Next steps:** encode categoricals, drop/derive redundant features, scale numeric features,
handle class imbalance, then train baseline models (Decision Tree, Random Forest, Logistic
Regression, KNN, Naive Bayes) followed by advanced models (XGBoost, SVM, Neural Network).
